In [1]:
##################
### Keagan Moo ###
### 2025/10/06 ###
### NoAge Peak ###
##################

# This script generates a peak set file that includes only those peaks that are
# not age associated via ARIMA or GLM in any cell type
# Also generates a file for each cell type with a strict (>0.5) threshold on
# association with age. Only confidently not age associated peaks get through

suppressPackageStartupMessages({library(tidyverse)
library(matrixStats)
library(dplyr)
library(ggrepel)
#library(optparse)
library(glue)
library(data.table)})

In [2]:
#Part 1 load in all types peaks individually  
message("Loading metadata")
metadata.factors <- readRDS("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts/metadata.factors")
peaks_dir <- file.path("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/Files/ARIMAke/",
                           "arushivData20220415/fusion_150_ccc")
Type1Peaks <- file.path(peaks_dir, glue("atac.Type_1.ccc.tsv")) %>%
 fread(data.table = F) %>%
 tibble::column_to_rownames("feature") %>%
 data.matrix()
Type1Peaks <- Type1Peaks[, colnames(Type1Peaks) != "V1"]
Type1Peaks <- Type1Peaks[,colnames(Type1Peaks) %in% rownames(metadata.factors)]
metadata.factors_Type1 <- metadata.factors[metadata.factors$sample %in% colnames(Type1Peaks),]
Type1Peaks <- Type1Peaks[,rownames(metadata.factors_Type1)]
Type1PeaksBed <- rownames(Type1Peaks)
Type1PeaksBedDF <- t(data.frame(strsplit(Type1PeaksBed, split = ":")))
rownames(Type1PeaksBedDF) <- NULL
colnames(Type1PeaksBedDF) <- NULL
Type1PeaksBedDF <- data.frame(Type1PeaksBedDF)
  
Type2aPeaks <- file.path(peaks_dir, glue("atac.Type_2a.ccc.tsv")) %>%
  fread(data.table = F) %>%
  tibble::column_to_rownames("feature") %>%
  data.matrix()
Type2aPeaks <- Type2aPeaks[, colnames(Type2aPeaks) != "V1"]
Type2aPeaks <- Type2aPeaks[,colnames(Type2aPeaks) %in% rownames(metadata.factors)]
metadata.factors_Type2a <- metadata.factors[metadata.factors$sample %in% colnames(Type2aPeaks),]
Type2aPeaks <- Type2aPeaks[,rownames(metadata.factors_Type2a)]
Type2aPeaksBed <- rownames(Type2aPeaks)
Type2aPeaksBedDF <- t(data.frame(strsplit(Type2aPeaksBed, split = ":")))
rownames(Type2aPeaksBedDF) <- NULL
colnames(Type2aPeaksBedDF) <- NULL
Type2aPeaksBedDF <- data.frame(Type2aPeaksBedDF)
 
Type2xPeaks <- file.path(peaks_dir, glue("atac.Type_2x.ccc.tsv")) %>%
  fread(data.table = F) %>%
  tibble::column_to_rownames("feature") %>%
  data.matrix()
Type2xPeaks <- Type2xPeaks[, colnames(Type2xPeaks) != "V1"]
Type2xPeaks <- Type2xPeaks[,colnames(Type2xPeaks) %in% rownames(metadata.factors)]
metadata.factors_Type2x <- metadata.factors[metadata.factors$sample %in% colnames(Type2xPeaks),]
Type2xPeaks <- Type2xPeaks[,rownames(metadata.factors_Type2x)]
Type2xPeaksBed <- rownames(Type2xPeaks)
Type2xPeaksBedDF <- t(data.frame(strsplit(Type2xPeaksBed, split = ":")))
rownames(Type2xPeaksBedDF) <- NULL
colnames(Type2xPeaksBedDF) <- NULL
Type2xPeaksBedDF <- data.frame(Type2xPeaksBedDF)

Loading metadata

Warning message in fread(., data.table = F):
“Detected 286 column names but the data has 287 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”
Warning message in fread(., data.table = F):
“Detected 286 column names but the data has 287 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”
Warning message in fread(., data.table = F):
“Detected 286 column names but the data has 287 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not 

In [3]:
#Part 2 load in all types GLMMNB peaks
  fType1_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/F_pTSS_Agg_Type_1.tsv"), sep = "\t", header = TRUE)
  mType1_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/M_pTSS_Agg_Type_1.tsv"), sep = "\t", header = TRUE)
  Type1_GLM <- rbind(fType1_GLM, mType1_GLM) %>% dplyr::select("estimate", "p_GLMMNB", "p_FDR") %>% dplyr::filter(p_FDR < 0.05)
  Type1_SelectGLM <- data.frame(Select = rownames(Type1_GLM))
  Type1_SigGLM <- unique(Type1_SelectGLM$Select)
  
  fType2a_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/F_pTSS_Agg_Type_2a.tsv"), sep = "\t", header = TRUE)
  mType2a_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/M_pTSS_Agg_Type_2a.tsv"), sep = "\t", header = TRUE)
  Type2a_GLM <- rbind(fType2a_GLM, mType2a_GLM) %>% dplyr::select("estimate", "p_GLMMNB", "p_FDR") %>% dplyr::filter(p_FDR < 0.05)
  Type2a_SelectGLM <- data.frame(Select = rownames(Type2a_GLM))
  Type2a_SigGLM <- unique(Type2a_SelectGLM$Select)
  
  fType2x_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/F_pTSS_Agg_Type_2x.tsv"), sep = "\t", header = TRUE)
  mType2x_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/M_pTSS_Agg_Type_2x.tsv"), sep = "\t", header = TRUE)
  Type2x_GLM <- rbind(fType2x_GLM, mType2x_GLM) %>% dplyr::select("estimate", "p_GLMMNB", "p_FDR") %>% dplyr::filter(p_FDR < 0.05)
  Type2x_SelectGLM <- data.frame(Select = rownames(Type2x_GLM))
  Type2x_SigGLM <- unique(Type2x_SelectGLM$Select)

In [8]:
#Part 3 establish one list of all age-associated peaks get some plots out of it

  Type1AgePeaks <- unique(c(Type1_SigGLM))
  Type2aAgePeaks <- unique(c(Type2a_SigGLM))
  Type2xAgePeaks <- unique(c(Type2x_SigGLM))
  
  AgePeaks <- unique(c(Type1AgePeaks, Type2aAgePeaks, Type2xAgePeaks))
  
  Type1PeaksBedDF$Col <- paste0(Type1PeaksBedDF$X1, ":", Type1PeaksBedDF$X2, ":", Type1PeaksBedDF$X3)
  Type1PeaksInter <- intersect(Type1PeaksBedDF$Col, AgePeaks)
  Type2aPeaksBedDF$Col <- paste0(Type2aPeaksBedDF$X1, ":", Type2aPeaksBedDF$X2, ":", Type2aPeaksBedDF$X3)
  Type2aPeaksInter <- intersect(Type2aPeaksBedDF$Col, AgePeaks)
  Type2xPeaksBedDF$Col <- paste0(Type2xPeaksBedDF$X1, ":", Type2xPeaksBedDF$X2, ":", Type2xPeaksBedDF$X3)
  Type2xPeaksInter <- intersect(Type2xPeaksBedDF$Col, AgePeaks)
  
  dfPlot <- data.frame(SetName = c("Type Peaks", "Age-associated Type Peaks", "Age-associated Total Peaks", "Type Total Overlap"),
                       Counts = c(nrow(Type1PeaksBedDF), length(Type1AgePeaks), length(AgePeaks), length(Type1PeaksInter)))
  
  pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/ATACEnrich/AgeAssociationProfile_Type_1.pdf"), width=8, height=6)
  mygg <- ggplot(dfPlot, aes(x = SetName, y = Counts)) +
    geom_bar(position = 'dodge', stat = "identity") +
    xlab("Peak Set") + ylab("Number of Peaks") + ggtitle(glue("Distribution of peaks and age-association (GLM + ARIMA)")) + 
    scale_fill_manual(values = c("#CC8000","#315CCC","#A9034C","#000000")) +
    scale_y_continuous(breaks=seq(0,200000,50000), limits = c(0,200000)) + 
    theme_bw()
  mygg
  dev.off()
  
  dfPlot <- data.frame(SetName = c("Type Peaks", "Age-associated Type Peaks", "Age-associated Total Peaks", "Type Total Overlap"),
                       Counts = c(nrow(Type2aPeaksBedDF), length(Type2aAgePeaks), length(AgePeaks), length(Type2aPeaksInter)))
  
  pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/ATACEnrich/AgeAssociationProfile_Type_2a.pdf"), width=8, height=6)
  mygg <- ggplot(dfPlot, aes(x = SetName, y = Counts)) +
    geom_bar(position = 'dodge', stat = "identity") +
    xlab("Peak Set") + ylab("Number of Peaks") + ggtitle(glue("Distribution of peaks and age-association (GLM + ARIMA)")) + 
    scale_fill_manual(values = c("#CC8000","#315CCC","#A9034C","#000000")) +
    scale_y_continuous(breaks=seq(0,200000,50000), limits = c(0,200000)) + 
    theme_bw()
  mygg
  dev.off()

  dfPlot <- data.frame(SetName = c("Type Peaks", "Age-associated Type Peaks", "Age-associated Total Peaks", "Type Total Overlap"),
                       Counts = c(nrow(Type2xPeaksBedDF), length(Type2xAgePeaks), length(AgePeaks), length(Type2xPeaksInter)))
  
  pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Plots/ATACEnrich/AgeAssociationProfile_Type_2x.pdf"), width=8, height=6)
  mygg <- ggplot(dfPlot, aes(x = SetName, y = Counts)) +
    geom_bar(position = 'dodge', stat = "identity") +
    xlab("Peak Set") + ylab("Number of Peaks") + ggtitle(glue("Distribution of peaks and age-association (GLM + ARIMA)")) + 
    scale_fill_manual(values = c("#CC8000","#315CCC","#A9034C","#000000")) +
    scale_y_continuous(breaks=seq(0,200000,50000), limits = c(0,200000)) + 
    theme_bw()
  mygg
  dev.off()

  saveRDS(AgePeaks, file = "/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/ChromEnrich/AgePeaks_FiberTypes_GLMMNB.rds")

pdf 
  2

pdf 
  2

pdf 
  2

In [12]:
#Part 4 Individual backgrounds for each with strict thresholds, trying (0.5, 0.1, 0.05, 0.01, 0.005)
    Type <- "Type_2x"
  fValFrame_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/F_pTSS_Agg_{Type}.tsv"), sep = "\t", header = TRUE)
  mValFrame_GLM <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/lmm/Agg/M_pTSS_Agg_{Type}.tsv"), sep = "\t", header = TRUE)
  ValFrame_GLM <- rbind(fValFrame_GLM, mValFrame_GLM) %>% dplyr::select("estimate", "p_GLMMNB", "p_FDR") %>% dplyr::filter(p_FDR > 0.5)
  ValFrame_AgeGLM <- unique(rownames(ValFrame_GLM))
  
  saveRDS(ValFrame_AgeGLM, file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240124_kgmoo_WolverineScripts/KMScripts_Update/Res/ChromEnrich/AgePeaks_{Type}_GLMMNB_05.rds"))